In [ ]:
# -*- coding: utf-8 -*-
"""
Merged script: geo_dis.ipynb + method_1.ipynb
Order: first compute geodesic distance, then run the complete method_1 pipeline.
"""

import os
import time
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.manifold import Isomap
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score

from scipy.sparse.csgraph import shortest_path
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon

warnings.filterwarnings("ignore")

os.makedirs("ours", exist_ok=True)


# ============================================================
# ================= geo_dis.ipynb content first ==============
# ============================================================

def compute_geodesic_distances_isomap(X, n_neighbors=15, n_components=10, sample_fraction=0.05):
    """
    Compute approximate geodesic distance matrix for high-dimensional data using Isomap.

    Parameters:
        X: (n_samples, n_features) data matrix
        n_neighbors: number of k-nearest neighbors
        n_components: Isomap embedding dimension (for acceleration)
        sample_fraction: subsampling ratio (0~1), use 0.05~0.1 if memory is limited

    Returns:
        geodesic_dist: (n_samples_sub, n_samples_sub) geodesic distance matrix
        sample_idx: original indices of the subsampled points
    """
    if sample_fraction < 1.0:
        n_samples = X.shape[0]
        n_sub = int(n_samples * sample_fraction)
        np.random.seed(42)
        sample_idx = np.random.choice(n_samples, n_sub, replace=False)
        X_sub = X[sample_idx, :]
        print(f"Subsampled to {n_sub} samples")
    else:
        X_sub = X
        sample_idx = np.arange(X.shape[0])
        n_sub = X_sub.shape[0]

    print("Building Isomap model...")
    start_time = time.time()

    isomap = Isomap(
        n_neighbors=n_neighbors,
        n_components=n_components,
        metric='euclidean',
        n_jobs=-1
    )
    embedding = isomap.fit_transform(X_sub)

    if hasattr(isomap, 'distance_matrix_'):
        geodesic_dist = isomap.distance_matrix_
    else:
        from sklearn.neighbors import kneighbors_graph
        knn_graph = kneighbors_graph(
            X_sub, n_neighbors=n_neighbors, mode='distance', include_self=False
        )
        geodesic_dist = shortest_path(csgraph=knn_graph, directed=False, method='D')

    elapsed = time.time() - start_time
    print(f"Geodesic distance computation completed in {elapsed:.2f} seconds")
    print(f"Distance matrix shape: {geodesic_dist.shape}")

    return geodesic_dist, sample_idx


sc_data_path = "dataGithub/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"

adata = sc.read_h5ad(sc_data_path)

geodesic_dist, sample_idx = compute_geodesic_distances_isomap(
    adata.X,
    n_neighbors=20,
    n_components=254,
    sample_fraction=1
)

print(f"Geodesic distance - Mean: {geodesic_dist.mean():.4f}, Std: {geodesic_dist.std():.4f}")

geodesic_npy_path = "dataGithub/geodesic_dist_sub.npy"
os.makedirs(os.path.dirname(geodesic_npy_path), exist_ok=True)
np.save(geodesic_npy_path, geodesic_dist, allow_pickle=True)


# ============================================================
# ================= method_1.ipynb content after =============
# ============================================================

def _sliced_wasserstein_distance(
    encoded_samples,
    distribution_samples,
    num_projections=50,
    p=2,
    device='cpu'
):
    """ Sliced Wasserstein Distance between encoded samples and drawn distribution samples. """
    embedding_dim = distribution_samples.size(1)
    projections = rand_projections(embedding_dim, num_projections).to(device)

    encoded_projections = encoded_samples.matmul(projections.transpose(0, 1).to(device))
    distribution_projections = (distribution_samples.matmul(projections.transpose(0, 1)))

    wasserstein_distance = (
        torch.sort(encoded_projections.transpose(0, 1), dim=1)[0]
        - torch.sort(distribution_projections.transpose(0, 1), dim=1)[0]
    )

    wasserstein_distance = torch.pow(wasserstein_distance, p)
    return wasserstein_distance.mean()


def corr_distance(x):
    # x: (n_samples, n_features)
    x_centered = x - x.mean(dim=0, keepdim=True)
    x_std = x_centered.std(dim=0, keepdim=True) + 1e-8
    x_norm = x_centered / x_std
    corr = (x_norm @ x_norm.T) / (x.shape[1] - 1)
    dist = 1 / 2 - corr / 2
    dist = torch.clamp(dist, min=0, max=1)
    return dist


def normalize_dist_matrix(D, method='median'):
    if method == 'median':
        scale = torch.median(D[D > 0])
    elif method == 'mean':
        scale = D.mean()
    else:
        scale = 1.0
    return D / (scale + 1e-8)


def compute_gromov_wasserstein_from_dist(z_cell, D_spot, A, p=2, gamma=1, normalize=True, epsilon=1e-8):
    """
    Compute Gromov-Wasserstein distance based on precomputed pairwise distance matrices.
    """
    n_cells, n_spots = A.shape

    if normalize:
        A_row = A / (A.sum(dim=1, keepdim=True) + epsilon)
        A_col = A_row / (A_row.sum(dim=0, keepdim=True) + epsilon)
        A = A_col

    row_sum = A.sum(dim=1, keepdim=True)
    col_sum = A.sum(dim=0, keepdim=True)
    D_cell = corr_distance(z_cell)

    term1 = ((D_cell ** p) * (row_sum @ row_sum.T)).sum()
    term2 = ((D_spot ** p) * (col_sum.T @ col_sum)).sum()
    term3 = -2 * torch.trace(A.T @ D_cell @ A @ D_spot)

    del D_cell
    loss = term1 + term2 + term3
    return loss / n_cells / n_spots


def geodesic_isometry_loss(z, sample_idx, geodesic_dist, method='corr', alpha=None):
    """
    Geodesic isometry constraint loss.
    """
    device = z.device

    z_sampled = z[sample_idx]
    n_sampled = z_sampled.shape[0]

    z_dist = torch.cdist(z_sampled, z_sampled, p=2)

    triu_idx = torch.triu_indices(n_sampled, n_sampled, offset=1)
    z_dist_flat = z_dist[triu_idx[0], triu_idx[1]]

    if isinstance(geodesic_dist, np.ndarray):
        geo_dist_tensor = torch.tensor(geodesic_dist, device=device, dtype=torch.float32)
    else:
        geo_dist_tensor = geodesic_dist.to(device)

    geo_dist_flat = geo_dist_tensor[triu_idx[0], triu_idx[1]]

    if method == 'mse':
        geo_dist_normalized = geo_dist_flat / (geo_dist_flat.max() + 1e-8)
        z_dist_normalized = z_dist_flat / (z_dist_flat.max() + 1e-8)
        loss = torch.mean((z_dist_normalized - geo_dist_normalized) ** 2)

    elif method == 'corr':
        z_mean = z_dist_flat.mean()
        geo_mean = geo_dist_flat.mean()
        z_centered = z_dist_flat - z_mean
        geo_centered = geo_dist_flat - geo_mean
        corr = (z_centered * geo_centered).sum() / (
            torch.sqrt((z_centered ** 2).sum() + 1e-8)
            * torch.sqrt((geo_centered ** 2).sum() + 1e-8)
        )
        loss = 1 / 2 - corr / 2

    elif method == 'contrastive':
        if alpha is None:
            alpha = (z_dist_flat / (geo_dist_flat + 1e-8)).median().detach()
        loss = torch.mean((z_dist_flat - alpha * geo_dist_flat) ** 2)

    else:
        raise ValueError(f"Unknown method: {method}")

    return loss


def pcc_distance_matrix(X_st, method='1-pcc'):
    """
    Compute the distance matrix between spots based on Pearson correlation.
    """
    corr = np.corrcoef(X_st, rowvar=True)
    corr = np.nan_to_num(corr, nan=0.0)

    if method == '1-pcc':
        D = (1 - corr) / 2
    elif method == 'sqrt':
        D = np.sqrt(2 * (1 - corr))
    else:
        raise ValueError("method must be '1-pcc' or 'sqrt'")

    np.fill_diagonal(D, 0.0)
    D = np.clip(D, 0, None)
    return torch.tensor(D, dtype=torch.float32)


def setup_device():
    """Set device to GPU if available, otherwise CPU."""
    return torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')


def create_similarity_graph(Y, k=15):
    """Create spatial similarity graph using Gaussian kernel on kNN distances."""
    n_spots = Y.shape[0]
    k = min(k, n_spots - 1)
    knn = NearestNeighbors(n_neighbors=k + 1).fit(Y)
    distances, indices = knn.kneighbors(Y)
    S = np.zeros((n_spots, n_spots))
    for i in range(n_spots):
        sigma = max(np.median(distances[i, 1:]), 1e-8)
        for j_idx, j in enumerate(indices[i]):
            if i != j:
                S[i, j] = np.exp(-distances[i, j_idx] ** 2 / (2 * sigma ** 2))
    if np.sum(S) == 0:
        S += np.random.random(S.shape) * 0.01
    return torch.tensor(S, dtype=torch.float32)


def find_highly_variable_genes(adata, n_top_genes=1000):
    """Find HVGs using Seurat v3 flavor; fallback to cell_ranger if needed."""
    try:
        sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes)
    except Exception:
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes)
    return np.where(adata.var['highly_variable'])[0]


def find_common_genes_from_hvg(adata_sc, adata_ST, n_top_genes=1000):
    """Return sorted list of common HVGs between sc and ST data."""
    adata_sc1 = adata_sc.copy()
    adata_ST1 = adata_ST.copy()
    sc_genes = set(adata_sc.var_names[find_highly_variable_genes(adata_sc1, n_top_genes)])
    st_genes = set(adata_ST.var_names[find_highly_variable_genes(adata_ST1, n_top_genes)])
    return sorted(sc_genes & st_genes)


def compute_scale_factor(X_sc, X_st):
    """Compute global scale factor = total ST counts / total sc counts."""
    return torch.sum(X_st) / torch.sum(X_sc)


def initialize_assignment_matrix(model, X_sc, X_st, n_neighbors=15):
    """Initialize A using kNN in PCA space (soft assignment)."""
    device = setup_device()
    X_sc_np = X_sc.cpu().numpy() if torch.is_tensor(X_sc) else X_sc
    X_st_np = X_st.cpu().numpy() if torch.is_tensor(X_st) else X_st
    combined = np.vstack([X_sc_np, X_st_np])
    combined = np.nan_to_num(combined, nan=0.0)
    row_sums = combined.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    normalized = combined / row_sums * row_sums.mean()
    pca = PCA(n_components=min(50, normalized.shape[1]))
    sc_pca = pca.fit_transform(normalized)[:X_sc_np.shape[0]]
    st_pca = pca.transform(normalized)[X_sc_np.shape[0]:]
    knn = NearestNeighbors(n_neighbors=min(n_neighbors, sc_pca.shape[0] - 1)).fit(sc_pca)
    _, indices = knn.kneighbors(st_pca)
    A_init = torch.zeros(model.n_st_spots, model.n_sc_cells)
    A_init = np.log(1e-10) * A_init
    for i in range(model.n_st_spots):
        A_init[i, indices[i]] = np.log(1.0 / len(indices[i]))
    A_init = torch.tensor(
        np.random.normal(0, 1, (model.n_st_spots, model.n_sc_cells)),
        dtype=torch.float32
    )
    model.A.data = A_init.to(device)
    return sc_pca


def load_and_align_data(sc_path, st_path, n_top_genes=1000, common_genes=None, fix_genes=False):
    """Load, filter, and align sc/ST data on common HVGs."""
    adata_ST = sc.read(st_path)
    adata_sc = sc.read(sc_path)
    sc.pp.normalize_total(adata_sc)

    if common_genes is None:
        common_genes = find_common_genes_from_hvg(adata_sc, adata_ST, n_top_genes)

    print('the number of common gene are', len(common_genes))
    assert len(common_genes) >= 100, "Too few common HVGs (<100)"

    if fix_genes == False:
        common_genes = list(set(common_genes) & set(adata_ST.var_names))
        common_genes = list(set(common_genes) & set(adata_sc.var_names))

    X_sc = adata_sc[:, common_genes].X.toarray() if hasattr(adata_sc.X, 'toarray') else adata_sc[:, common_genes].X
    X_st = adata_ST[:, common_genes].X.toarray() if hasattr(adata_ST.X, 'toarray') else adata_ST[:, common_genes].X

    X_sc, X_st = torch.tensor(X_sc, dtype=torch.float32), torch.tensor(X_st, dtype=torch.float32)
    Y = torch.tensor(adata_ST.obsm['spatial'], dtype=torch.float32) if 'spatial' in adata_ST.obsm else None

    return X_sc, X_st, Y, adata_sc, adata_ST, common_genes


class VAEImprovedSpatialModel(nn.Module):
    def __init__(
        self, n_sc_cells, n_st_spots, n_genes, embedding_dim=8,
        lambda_reg=0, lambda_l2=0, lambda_l4=0, lambda_l5=0, lambda_ot=0,
        lambda_M=1, lambda_M1=0, lambda_r=0, lambda_kl=0, lambda_pca=0,
        lambda_f=0, lambda_row_sum=0, predictA=None, lambda_g=0
    ):
        super().__init__()
        self.n_sc_cells = n_sc_cells
        self.n_st_spots = n_st_spots
        self.n_genes = n_genes
        self.embedding_dim = embedding_dim
        self.lambda_reg = lambda_reg
        self.lambda_l2 = lambda_l2
        self.lambda_l4 = lambda_l4
        self.lambda_l5 = lambda_l5
        self.lambda_ot = lambda_ot
        self.lambda_r = lambda_r
        self.lambda_M = lambda_M
        self.lambda_g = lambda_g
        self.lambda_M1 = lambda_M1
        self.lambda_kl = lambda_kl
        self.lambda_pca = lambda_pca
        self.lambda_f = lambda_f
        self.lambda_row_sum = lambda_row_sum
        self._density_criterion = torch.nn.KLDivLoss(reduction="mean")

        self.encoder = nn.Sequential(
            nn.Linear(n_genes, 128),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64), nn.ReLU()
        )
        self.predictA = predictA
        self.fc_mu = nn.Linear(64, embedding_dim)
        self.fc_logvar = nn.Linear(64, embedding_dim)
        self.logit_pi = nn.Parameter(torch.tensor(1.386))

        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, n_genes),
            nn.Softplus()
        )
        self.align_mlp = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim), nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim)
        )

        device = setup_device()
        self.A = np.random.normal(0, 1, (n_st_spots, n_sc_cells))
        self.A = nn.Parameter(
            torch.tensor(self.A, device=device, requires_grad=True, dtype=torch.float32)
        )
        self.f = nn.Parameter(torch.randn(n_sc_cells, 1, device=device))
        self.log_scale = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))
        self.gamma1 = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        self.gamma2 = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode_sc(self, X_sc):
        h = self.encoder(X_sc)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10, max=10)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def decode_expression(self, z_cell):
        return self.decoder(z_cell)

    def kl_loss(self, mu, logvar):
        return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    def compute_softmax_A(self):
        return F.softmax(self.A, dim=0)

    def compute_st_embedding(self, z_cell):
        A_hat = self.compute_softmax_A()
        return torch.mm(A_hat, z_cell)

    def poisson_nll_loss(self, y_true, lam):
        lam = torch.clamp(lam, min=1e-8)
        return F.poisson_nll_loss(
            input=torch.log(lam + 1e-16),
            target=y_true,
            log_input=True,
            full=False,
            reduction='mean'
        )

    def zero_inflated_poisson_nll_loss(self, y_true, lam, eps=1e-8):
        lam = torch.clamp(lam, min=1e-8)
        pi = torch.sigmoid(self.logit_pi)

        log_p_poisson = y_true * torch.log(lam) - lam - torch.lgamma(y_true + 1)
        p_poisson = torch.exp(log_p_poisson)

        is_zero = (y_true == 0).float()
        prob = is_zero * (pi + (1 - pi) * torch.exp(-lam)) + \
               (1 - is_zero) * ((1 - pi) * p_poisson)

        nll = -torch.log(prob + eps)
        return nll.mean()

    def loss_L1(self, X_sc, f_vals):
        return self.poisson_nll_loss(X_sc, f_vals)

    def loss_L1_cos(self, X_sc, f_vals):
        gv = 1 - 0.5 * torch.nn.functional.cosine_similarity(f_vals, X_sc, dim=0).mean()
        return gv

    def loss_L2(self, X_st, f_vals, A_softmax=None):
        if self.lambda_f == 0:
            X_st_pred = torch.mm(A_softmax, f_vals)
        else:
            f_sigmoid = torch.sigmoid(self.f)
            s_vec = f_sigmoid.squeeze()
            if f_vals.dim() == 1:
                weighted_vec = s_vec * f_vals
            else:
                weighted_vec = s_vec.unsqueeze(-1) * f_vals
            X_st_pred = torch.mm(A_softmax, weighted_vec)
        return self.poisson_nll_loss(X_st, self.log_scale * X_st_pred)

    def loss_L3(self, z_st, S):
        diff = z_st.unsqueeze(1) - z_st.unsqueeze(0)
        sq_diff = torch.mean(diff ** 2, dim=2)
        return torch.mean(S * sq_diff) if torch.sum(S) > 0 else torch.sum(sq_diff) * 1e-6

    def loss_LM(self, X_sc, X_st):
        A_softmax = self.compute_softmax_A()
        if self.lambda_f == 0:
            G_pred = torch.matmul(A_softmax, X_sc)
        else:
            f_sigmoid = torch.sigmoid(self.f)
            weighted_X_sc = f_sigmoid * X_sc
            G_pred = torch.matmul(A_softmax, weighted_X_sc)

        gv = 1 / 2 - 0.5 * torch.nn.functional.cosine_similarity(G_pred, X_st, dim=0).mean()
        return gv

    def loss_LM1(self, X_sc, X_st, mapping_original):
        A_softmax = self.compute_softmax_A()
        if self.lambda_f == 0:
            G_pred = torch.matmul(A_softmax, X_sc)
        else:
            f_sigmoid = torch.sigmoid(self.f)
            weighted_X_sc = f_sigmoid * X_sc
            G_pred = torch.matmul(A_softmax, weighted_X_sc)

        scaled_pred = torch.exp(self.log_scale) * G_pred

        if mapping_original == 'original':
            return self.zero_inflated_poisson_nll_loss(
                torch.log(1 + X_st), torch.log(1 + scaled_pred)
            )
        else:
            return self.zero_inflated_poisson_nll_loss(X_st, scaled_pred)

    def custom_loss(self):
        f_sigmoid = torch.sigmoid(self.f)
        return torch.mean(f_sigmoid - f_sigmoid ** 2)

    def row_sum_mse_loss(self, num_cell):
        A_softmax = self.compute_softmax_A()

        if self.lambda_f == 0:
            A = A_softmax
        else:
            sigmoid_f = torch.sigmoid(self.f).squeeze()
            A = A_softmax * sigmoid_f

        row_sums = A.sum(dim=1) / A.sum()
        row_sums = torch.log(row_sums)
        num_cell = num_cell / num_cell.sum()

        density_term = self._density_criterion(
            row_sums.unsqueeze(0), num_cell.unsqueeze(0)
        )
        return density_term

    def forward(
        self, X_sc, X_st, z_pca=None, S=None, D=None, num_cell=None,
        f_vals=None, z_cell=None, sample_idx=None, geodesic_dist=None,
        mapping_original=None
    ):
        target_dtype = next(self.parameters()).dtype

        def ensure_dtype(tensor):
            if tensor is None:
                return None
            if tensor.dtype != target_dtype:
                return tensor.to(target_dtype)
            return tensor

        X_sc = ensure_dtype(X_sc)
        X_st = ensure_dtype(X_st)

        if self.predictA is not None:
            S = ensure_dtype(S)
            num_cell = ensure_dtype(num_cell)

            A_hat = self.compute_softmax_A()
            l3 = compute_gromov_wasserstein_from_dist(
                z_cell, D, A_hat.T, gamma=self.gamma2
            )
            LM = self.loss_LM(X_sc, X_st)
            LM1 = self.loss_LM1(f_vals, X_st, mapping_original)
            l_f = self.custom_loss()
            del A_hat

            if num_cell is not None:
                row_sum_loss = self.row_sum_mse_loss(num_cell)

            total = (
                self.lambda_M * LM
                + self.lambda_M1 * LM1
                + self.lambda_f * l_f
                + self.lambda_row_sum * row_sum_loss
                + self.lambda_reg * l3
            )
            return total, LM, LM1, l_f, row_sum_loss, l3

        else:
            z_cell, mu, logvar = self.encode_sc(X_sc)
            f_vals = self.decode_expression(z_cell)
            l1 = self.loss_L1(X_sc, f_vals)
            kl = self.kl_loss(mu, logvar)
            l4 = self.loss_L1_cos(X_sc, f_vals)

            if z_pca is None:
                raise ValueError("z_pca must be provided for alignment loss.")

            z_cell_proj = self.align_mlp(z_cell)
            z_pca = torch.tensor(z_pca, device=X_sc.device, dtype=torch.float32)
            loss_align = F.mse_loss(z_cell_proj, z_pca, reduction='mean')
            l_g = geodesic_isometry_loss(
                z_cell, sample_idx, geodesic_dist, method='corr', alpha=None
            )

            total = (
                l1
                + self.lambda_pca * loss_align
                + self.lambda_kl * kl
                + self.lambda_l4 * l4
                + self.lambda_g * l_g
            )
            return total, l1, l4, kl, loss_align, l_g


def train_vae_poisson_model(
    sc_path, st_path, C=None, num_cell=None, output_dir='SpatialVG_VAE_Poisson',
    common_genes=None, embedding_dim=8, lr=0.01, epochs=300,
    lambda_reg=0.5, lambda_l2=1, lambda_l4=1, lambda_l5=1, lambda_ot=-1,
    lambda_M=1, lambda_M1=1, lambda_r=-10000, lambda_kl=0.1, lambda_pca=1,
    lambda_f=1, lambda_row_sum=1.0, lambda_g=1, n_top_genes=1000,
    z_pca=None, predictA=None, f_vals=None, z_cell=None, fix_genes=False,
    geodesic_dist=None, sample_idx=None, mapping_original=None
):
    device = setup_device()
    os.makedirs(f'{output_dir}/models', exist_ok=True)
    os.makedirs(f'{output_dir}/evaluation', exist_ok=True)

    X_sc, X_st, Y, adata_sc, adata_ST, genes = load_and_align_data(
        sc_path, st_path, n_top_genes=n_top_genes,
        common_genes=common_genes, fix_genes=fix_genes
    )

    X_sc = np.log(X_sc + 1)
    X_st = np.log(X_st + 1)

    if predictA is not None:
        S = create_similarity_graph(Y.numpy()) if Y is not None else torch.zeros(
            X_st.shape[0], X_st.shape[0]
        )
        S = S.to(device)
        D = pcc_distance_matrix(X_st, method='1-pcc')
        D = D.to(device)
        S = normalize_dist_matrix(1 - S, 'median')
        D = normalize_dist_matrix(D, 'median')

    X_sc, X_st = X_sc.to(device), X_st.to(device)
    print(f"Training on device: {device}")

    model = VAEImprovedSpatialModel(
        n_sc_cells=X_sc.shape[0],
        n_st_spots=X_st.shape[0],
        n_genes=X_sc.shape[1],
        embedding_dim=embedding_dim,
        lambda_reg=lambda_reg,
        lambda_l2=lambda_l2,
        lambda_l4=lambda_l4,
        lambda_l5=lambda_l5,
        lambda_ot=lambda_ot,
        lambda_M=lambda_M,
        lambda_M1=lambda_M1,
        lambda_r=lambda_r,
        lambda_kl=lambda_kl,
        lambda_pca=lambda_pca,
        lambda_f=lambda_f,
        lambda_row_sum=lambda_row_sum,
        lambda_g=lambda_g,
        predictA=predictA
    ).to(device)

    if z_pca is None:
        pca_sc = initialize_assignment_matrix(model, X_sc, X_st, n_neighbors=15)
    else:
        pca_sc = z_pca.copy()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.8)

    best_loss, patience = float('inf'), 0
    losses = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        if predictA is None:
            total, l1, l2, kl, loss_align, l_g = model(
                X_sc, X_st,
                z_pca=pca_sc[:, :embedding_dim],
                geodesic_dist=geodesic_dist,
                sample_idx=sample_idx
            )
        else:
            if mapping_original == 'original':
                total, LM, LM1, l_f, row_sum_loss, l3 = model(
                    torch.exp(X_sc) - 1,
                    torch.exp(X_st) - 1,
                    S=S, D=D, num_cell=num_cell,
                    f_vals=torch.exp(f_vals) - 1,
                    z_cell=z_cell,
                    mapping_original=mapping_original
                )
            else:
                total, LM, LM1, l_f, row_sum_loss, l3 = model(
                    X_sc, X_st,
                    S=S, D=D, num_cell=num_cell,
                    f_vals=f_vals, z_cell=z_cell
                )

        if torch.isnan(total).any():
            print(f"NaN detected at epoch {epoch}. Stopping training.")
            break

        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if predictA is None:
            losses.append({
                'epoch': epoch,
                'total': total.item(),
                'l1': l1.item(),
                'kl': kl.item(),
                'loss_align': loss_align.item()
            })
        else:
            losses.append({
                'epoch': epoch,
                'total': total.item(),
                'LM': LM.item(),
                'LM1': LM1.item(),
                'l_f': l_f.item(),
                'row_sum_loss': row_sum_loss.item(),
                'l3': l3.item()
            })

        if total.item() < best_loss:
            best_loss, patience = total.item(), 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'common_genes': genes,
                'config': {
                    'n_sc_cells': X_sc.shape[0],
                    'n_st_spots': X_st.shape[0],
                    'n_genes': X_sc.shape[1],
                    'embedding_dim': embedding_dim,
                    'lambda_reg': lambda_reg,
                    'lambda_l2': lambda_l2,
                    'lambda_l4': lambda_l4,
                    'lambda_l5': lambda_l5,
                    'lambda_ot': lambda_ot,
                    'lambda_M': lambda_M,
                    'lambda_r': lambda_r,
                    'lambda_kl': lambda_kl,
                    'loss_align': lambda_pca,
                    'loss_f': lambda_f,
                    'lambda_row_sum': lambda_row_sum
                }
            }, f'{output_dir}/models/best_vae_poisson_model.pth')
        else:
            patience += 1
            if patience >= 200:
                break

        if epoch % 20 == 0:
            if predictA is None:
                print(
                    f'Epoch {epoch:3d}: Total={total:.2f}, L1={l1:.2f}, '
                    f'L4={lambda_l4*l2:.2f}, KL={lambda_kl*kl:.2f}, '
                    f'PCA={lambda_pca*loss_align:.2f},geo={lambda_g*l_g:.2f}'
                )
            else:
                print(
                    f'Epoch {epoch:3d}: Total={total:.2f}, LM={LM:.2f}, '
                    f'LM1={LM1:.2f}, l_f={l_f:.2f}, '
                    f'row_sum_loss={10000*row_sum_loss:.2f},l3={l3:.2f}'
                )

    return model, genes, losses


def build_A_gt_from_spot_to_cells(adata_st, adata_sc, spot_to_cells):
    """
    Build ground truth assignment matrix A_gt (n_spots x n_cells).
    """
    n_spots = adata_st.n_obs
    n_cells = adata_sc.n_obs
    A_gt = np.zeros((n_spots, n_cells), dtype=np.float32)

    for i, spot_name in enumerate(adata_st.obs_names):
        if spot_name in spot_to_cells:
            cell_indices = spot_to_cells[spot_name]
            for cid in cell_indices:
                if 0 <= cid < n_cells:
                    A_gt[i, cid] = 1.0
                else:
                    raise ValueError(f"Cell index {cid} out of range for {n_cells} cells")
    return A_gt


def evaluate_predictions_numpy(
    predictsc,
    scembeding,
    X_sc_true,
    predictst=None,
    X_st_true=None,
    predictA=None,
    A_gt=None,
    cell_type_labels=None,
    A_gt_deconv=None,
    cell_types_gt_order=None
):
    """
    Pure numpy evaluation.
    Some inputs can be None.
    """
    metrics = {}

    if predictsc is not None:
        predictsc = np.asarray(predictsc, dtype=np.float32)
    if X_sc_true is not None:
        X_sc_true = np.asarray(X_sc_true, dtype=np.float32)
    if predictst is not None:
        predictst = np.asarray(predictst, dtype=np.float32)
    if X_st_true is not None:
        X_st_true = np.asarray(X_st_true, dtype=np.float32)
    if predictA is not None:
        predictA = np.asarray(predictA, dtype=np.float32)
    if A_gt is not None:
        A_gt = np.asarray(A_gt, dtype=np.float32)

    def mean_pcc(x_true, x_pred):
        row_pccs, col_pccs = [], []
        for i in range(x_true.shape[0]):
            t, p = x_true[i], x_pred[i]
            if np.std(t) > 1e-8 and np.std(p) > 1e-8:
                r, _ = pearsonr(t, p)
                row_pccs.append(float(r))
            else:
                row_pccs.append(0.0)
        for j in range(x_true.shape[1]):
            t, p = x_true[:, j], x_pred[:, j]
            if np.std(t) > 1e-8 and np.std(p) > 1e-8:
                r, _ = pearsonr(t, p)
                col_pccs.append(float(r))
            else:
                col_pccs.append(0.0)
        return np.mean(row_pccs), np.mean(col_pccs)

    if predictsc is not None and X_sc_true is not None:
        f_row_pcc, f_col_pcc = mean_pcc(X_sc_true, predictsc)
        metrics['predictsc_row_pcc'] = float(f_row_pcc)
        metrics['predictsc_col_pcc'] = float(f_col_pcc)

    if predictst is not None and X_st_true is not None:
        st_row_pcc, st_col_pcc = mean_pcc(X_st_true, predictst)
        metrics['predictst_row_pcc'] = float(st_row_pcc)
        metrics['predictst_col_pcc'] = float(st_col_pcc)

    if predictA is not None and A_gt is not None:
        pred_assign = np.argmax(predictA, axis=1)
        true_assign = np.argmax(A_gt, axis=1)
        metrics['assignment_accuracy'] = float(np.mean(pred_assign == true_assign))

    dec_pred_aligned = None

    if predictA is not None and cell_type_labels is not None:
        le = LabelEncoder()
        labels = le.fit_transform(cell_type_labels)
        n_types = len(le.classes_)
        one_hot_sc = np.eye(n_types)[labels]
        dec_pred_raw = predictA @ one_hot_sc
        dec_pred_raw = dec_pred_raw / (dec_pred_raw.sum(axis=1, keepdims=True))

        if A_gt_deconv is not None:
            A_gt_deconv = np.asarray(A_gt_deconv, dtype=np.float32)
            pred_order = le.classes_.tolist()
            gt_order = cell_types_gt_order
            reorder_idx = [pred_order.index(ct) for ct in gt_order]
            dec_pred_aligned = dec_pred_raw[:, reorder_idx]

            pcc_list = []
            for i in range(A_gt_deconv.shape[0]):
                gt_i = A_gt_deconv[i]
                pred_i = dec_pred_aligned[i]
                if np.std(gt_i) < 1e-8 or np.std(pred_i) < 1e-8:
                    r = 0.0
                else:
                    r, _ = pearsonr(gt_i, pred_i)
                    r = float(r) if np.isfinite(r) else 0.0
                pcc_list.append(r)

            metrics['deconv_mean_pcc'] = float(np.mean(pcc_list))
            metrics['deconv_mean_mae'] = float(np.mean(np.abs(A_gt_deconv - dec_pred_aligned)))
            metrics['deconv_mean_rmse'] = float(np.sqrt(np.mean((A_gt_deconv - dec_pred_aligned) ** 2)))
            metrics['PCC_all'] = pcc_list

    if scembeding is not None and cell_type_labels is not None:
        le = LabelEncoder()
        y = le.fit_transform(cell_type_labels)
        if len(np.unique(y)) > 1:
            correlation_distances = pdist(scembeding, metric='correlation')
            distance_matrix = squareform(correlation_distances)
            sil = silhouette_score(distance_matrix, y, metric='precomputed')
            metrics['silhouette_index'] = float(sil)
        else:
            metrics['silhouette_index'] = 0.0

    print("✅ Evaluation complete.")

    if predictsc is not None and X_sc_true is not None and cell_type_labels is not None:
        predictsc = np.asarray(predictsc)
        X_sc_true = np.asarray(X_sc_true)
        cell_type_labels = np.asarray(cell_type_labels)
        unique_types = np.unique(cell_type_labels)
        unique_types = unique_types[:10]
        n_types = len(unique_types)

        n_cols = 5
        n_rows = int(np.ceil(n_types / n_cols))

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(4 * n_cols, 4 * n_rows),
            squeeze=False
        )

        idx = -1
        for idx, ct in enumerate(unique_types):
            row = idx // n_cols
            col = idx % n_cols
            ax = axes[row, col]

            ct_indices = np.where(cell_type_labels == ct)[0]
            if len(ct_indices) == 0:
                ax.axis("off")
                continue

            cell_idx = ct_indices[0]
            x = X_sc_true[cell_idx]
            y = predictsc[cell_idx]

            if np.std(x) > 1e-8 and np.std(y) > 1e-8:
                r, _ = pearsonr(x, y)
            else:
                r = 0.0

            ax.scatter(x, y, s=6, alpha=0.6)
            ax.set_title(f"{ct}\nPCC={r:.2f}", fontsize=10)
            ax.set_xlabel("X_sc_true")
            ax.set_ylabel("predictsc")

        for j in range(idx + 1, n_rows * n_cols):
            axes[j // n_cols, j % n_cols].axis("off")

        plt.tight_layout()
        plt.show()

    return metrics, dec_pred_aligned


# ============================================================
# Read the initial data for method_1
# ============================================================

adata_sc = sc.read(
    "dataGithub/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
)
common_genes = adata_sc.var_names

geodesic_dist = np.load(
    'dataGithub/geodesic_dist_sub.npy',
    allow_pickle=True
)

sc_data_path = "dataGithub/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
st_data_path = "dataGithub/adata_st_mouse1sample1_mouse1_slice50_drop.h5ad"

count_file_path = "dataGithub/spots_info1.csv"
Count_cell = pd.read_csv(count_file_path)


# ============================================================
# First training: predictA=None
# ============================================================

num_cell = Count_cell['n_cells'].values
num_cell = torch.tensor(num_cell, device='cuda:2')

print("=" * 60)
print("Starting improved model training")
print("=" * 60)

model, common_genes, losses = train_vae_poisson_model(
    sc_path=sc_data_path,
    st_path=st_data_path,
    num_cell=num_cell,
    output_dir='SpatialVG_improved_NMF',
    common_genes=common_genes,
    embedding_dim=32,
    lr=0.05,
    epochs=500,
    lambda_g=1,
    lambda_reg=0,
    lambda_l2=0,
    lambda_l4=1,
    lambda_l5=0.0000000,
    lambda_ot=0,
    lambda_M=0,
    lambda_M1=0,
    lambda_r=0.0000,
    lambda_kl=0.0001,
    lambda_pca=0,
    lambda_f=0,
    lambda_row_sum=0,
    n_top_genes=5000,
    z_pca=None,
    geodesic_dist=geodesic_dist,
    sample_idx=np.arange(geodesic_dist.shape[0]),
    fix_genes=True
)

os.makedirs('SpatialVG_improved_NMF-train-test/models', exist_ok=True)
os.makedirs('SimMerfish/ours', exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'common_genes': common_genes,
    'training_losses': losses,
}, 'SimMerfish/ours/no_pca_pred_zcell.pth')

print("Training completed and model saved.")


# ============================================================
# performance / first model encoding and prediction
# ============================================================

sc_path = "dataGithub/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
st_path = "dataGithub/adata_st_mouse1sample1_mouse1_slice50_drop.h5ad"

n_top_genes = 5000

X_sc, X_st, Y, adata_sc, adata_ST, common_genes = load_and_align_data(
    sc_path, st_path, n_top_genes,
    common_genes=common_genes, fix_genes=True
)

X_sc = np.log(X_sc + 1)
X_st = np.log(X_st + 1)

n_sc_cells = X_sc.shape[0]
n_st_spots = X_st.shape[0]
n_genes = len(common_genes)
embedding_dim = 64

adata_sc = adata_sc[:, common_genes]
adata_st = adata_ST[:, common_genes]

model.eval()
device = next(model.parameters()).device

X_sc_tensor = X_sc.to(device).float()

with torch.no_grad():
    z_cell, _, _ = model.encode_sc(X_sc_tensor)
    f_vals = model.decoder(z_cell)

    A_hat = model.compute_softmax_A()
    scale = torch.exp(model.log_scale)

    X_st_pred = torch.mm(scale * A_hat, f_vals)
    sigmoid_f = torch.sigmoid(model.f)

predictsc = f_vals.cpu().numpy()
X_st_pred = X_st_pred.cpu().numpy()
A_hat_np = A_hat.cpu().numpy()
scembeding = z_cell.cpu().numpy()

metadata = adata_sc.obs
cell_type_labels = metadata['celltype'].values

A_hat_np = pd.DataFrame(A_hat_np, index=adata_st.obs_names, columns=adata_sc.obs_names)

os.makedirs('ours', exist_ok=True)
np.save('ours/predictsc.npy', predictsc)
np.save('ours/scembeding.npy', scembeding)


# ============================================================
# First evaluation
# ============================================================

result_ours = evaluate_predictions_numpy(
    predictsc,
    scembeding,
    np.log(1 + adata_sc.X),
    predictst=None,
    X_st_true=np.log(1 + adata_st.X.toarray()),
    predictA=None,
    A_gt=None,
    cell_type_labels=cell_type_labels,
    A_gt_deconv=None,
    cell_types_gt_order=None
)


# ============================================================
# Louvain clustering evaluation
# ============================================================

z_np = z_cell.cpu().numpy() if z_cell.is_cuda else z_cell.numpy()
true_labels = adata_sc.obs['celltype'].values

print(f"Embedding shape: {z_np.shape}")
print(f"Number of labels: {len(true_labels)}")
print(f"Number of true cell types: {len(np.unique(true_labels))}")

adata = sc.AnnData(z_np)
adata.obs['celltype'] = true_labels

n_neighbors = 20
sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep='X')

resolutions = [0.2, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0]

results = []
best_nmi = -1
best_ari = -1
best_res = None

for res in resolutions:
    sc.tl.louvain(adata, resolution=res, key_added=f'louvain_r{res}')
    pred_labels = adata.obs[f'louvain_r{res}'].values

    ari = adjusted_rand_score(true_labels, pred_labels)
    nmi = normalized_mutual_info_score(true_labels, pred_labels)

    results.append({'resolution': res, 'ARI': ari, 'NMI': nmi})

    if nmi > best_nmi:
        best_nmi = nmi
        best_ari = ari
        best_res = res

    print(f"Resolution {res:.1f} -> ARI: {ari:.4f}, NMI: {nmi:.4f}")

print("\n" + "=" * 50)
print(f"Best resolution: {best_res}")
print(f"Best NMI: {best_nmi:.4f}")
print(f"Corresponding ARI: {best_ari:.4f}")
print("=" * 50)


# ============================================================
# Save first model results
# ============================================================

pd.DataFrame(scembeding).to_csv('ours/merfish_sc_embeding.csv')
pd.DataFrame(predictsc).to_csv('ours/merfish_sc_pred_0425.csv')
pd.DataFrame(np.log(1 + adata_sc.X)).to_csv('ours/merfish_sc_groundtruth_0425.csv')

print(result_ours)


# ============================================================
# fix the encoder result and optimize A
# Second training: predictA=True
# ============================================================

adata_sc = sc.read(
    "dataGithub/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
)
common_genes = adata_sc.var_names
device = 'cuda:2'

predictsc = np.load(
    'SimMerfish/ours/predictsc.npy'
)
scembeding = np.load(
    'SimMerfish/ours/scembeding.npy'
)

sc_path = "dataGithub/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
st_path = "dataGithub/adata_st_mouse1sample1_mouse1_slice50_drop.h5ad"

sc_data_path = sc_path
st_data_path = st_path

count_file_path = 'dataGithub/spots_info2.csv'

checkpoint_path = 'SimMerfish/ours/no_pca_pred_zcell.pth'
checkpoint = torch.load(checkpoint_path, map_location='cuda:2', weights_only=False)

Count_cell = pd.read_csv(count_file_path)
num_cell = Count_cell['n_cells'].values
num_cell = torch.tensor(num_cell, device='cuda:2')

print("=" * 60)
print("Starting improved model training")
print("=" * 60)

model, common_genes, losses = train_vae_poisson_model(
    sc_path=sc_data_path,
    st_path=st_data_path,
    num_cell=num_cell,
    output_dir='SpatialVG_improved_NMF',
    common_genes=common_genes,
    embedding_dim=32,
    lr=0.05,
    epochs=500,
    lambda_reg=0.1,
    lambda_l2=0.01,
    lambda_l4=0,
    lambda_l5=0.0000000,
    lambda_ot=0,
    lambda_M=1,
    lambda_M1=1,
    lambda_r=0.0000,
    lambda_kl=0.000,
    lambda_pca=0,
    lambda_f=0,
    lambda_row_sum=10,
    n_top_genes=5000,
    z_pca=None,
    predictA=True,
    f_vals=torch.tensor(predictsc, device='cuda:2'),
    z_cell=torch.tensor(scembeding, device='cuda:2'),
    fix_genes=True,
    mapping_original='original'
)

os.makedirs('SpatialVG_improved_NMF/models', exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'common_genes': common_genes,
    'training_losses': losses,
}, 'SpatialVG_improved_NMF/models/fix_enc_harm_getA2.pth')

print("Training completed and model saved.")


# ============================================================
# Read deconvolution ground truth
# ============================================================

A_gt_deconv = pd.read_csv(
    'dataGithub/A_gt_simulationdata.csv',
    sep=',', header=0, index_col=0
)

metadata = adata_sc.obs
cell_type_labels = metadata['celltype'].values


# ============================================================
# Reload data and perform the second model evaluation
# ============================================================

sc_path = "dataGithub/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
st_path = "dataGithub/adata_st_mouse1sample1_mouse1_slice50_drop.h5ad"

n_top_genes = 5000

X_sc, X_st, Y, adata_sc, adata_ST, common_genes = load_and_align_data(
    sc_path, st_path, n_top_genes,
    common_genes=common_genes, fix_genes=True
)

X_sc = np.log(X_sc + 1)
X_st = np.log(X_st + 1)

adata_st = adata_ST

model.eval()
device = next(model.parameters()).device

X_sc_tensor = X_sc.to(device).float()

with torch.no_grad():
    sigmoid_f = torch.sigmoid(model.f).squeeze()
    A_hat = model.compute_softmax_A()

A_hat_np = A_hat.cpu().numpy()
stembeding = A_hat_np @ scembeding
scale = torch.exp(model.log_scale)

predictst = scale.cpu().detach().numpy() * A_hat_np @ (np.exp(predictsc) - 1)

A_hat_np = pd.DataFrame(A_hat_np, index=adata_st.obs_names, columns=adata_sc.obs_names)
A_hat_np.index = A_hat_np.index.astype(int)
A_hat_np = A_hat_np.loc[A_gt_deconv.index, :]

cell_types_gt_order = A_gt_deconv.columns.tolist()

pd.DataFrame(stembeding).to_csv('merfish_stembeding.csv')


# ============================================================
# Build A_gt
# ============================================================

cell_spot_gt = pd.read_csv(
    'dataGithub/spots_cells.csv',
    sep=',', header=0, index_col=0
)

matrix = pd.get_dummies(cell_spot_gt['spot_id']).astype(int)
A_gt = matrix.copy()
A_gt.columns = A_gt.columns.astype(str)
A_gt = A_gt[adata_ST.obs.index]
A_gt_reindexed = A_gt.reindex(adata_sc.obs_names, fill_value=0)


# ============================================================
# Second full evaluation
# ============================================================

result_ours = evaluate_predictions_numpy(
    predictsc,
    None,
    np.log(1 + adata_sc.X),
    predictst=np.log(1 + predictst),
    X_st_true=np.log(1 + adata_st.X),
    predictA=A_hat_np.values,
    A_gt=A_gt_reindexed.T,
    cell_type_labels=cell_type_labels,
    A_gt_deconv=A_gt_deconv,
    cell_types_gt_order=cell_types_gt_order
)

print(result_ours)

A_hat_np.to_csv('ours/predicted_A_matrix-0425.csv')


# ============================================================
# Entropy filtering + deconvolution evaluation
# ============================================================

def get_entropy_uniformity(matrix):
    epsilon = 1e-12
    matrix = np.clip(matrix, epsilon, 1)

    N = matrix.shape[0]
    entropy = -np.sum(matrix * np.log(matrix), axis=0)
    max_entropy = np.log(N)
    uniformity_score = entropy / max_entropy

    return uniformity_score


A_hat_np = A_hat_np / A_hat_np.sum(axis=0)
AA = get_entropy_uniformity(A_hat_np)

index0 = np.where(AA < 0.8)[0]
A_hat_np1 = A_hat_np.iloc[:, index0]

result_ours, A_dec_pred = evaluate_predictions_numpy(
    None,
    None,
    np.log(1 + adata_sc.X),
    predictst=None,
    X_st_true=np.log(1 + adata_st.X),
    predictA=A_hat_np1,
    A_gt=None,
    cell_type_labels=cell_type_labels[index0],
    A_gt_deconv=A_gt_deconv,
    cell_types_gt_order=cell_types_gt_order
)

print(result_ours)

A_dec_pred1 = A_dec_pred.copy()
A_dec_pred1 = A_dec_pred1.T / A_dec_pred1.sum(axis=1)
A_dec_pred1 = A_dec_pred1.T

AA = get_entropy_uniformity(A_dec_pred1.T)

index1 = np.where(AA < 0.9)[0]
print(len(index1))

pcc_list = []
for i in range(A_gt_deconv.iloc[index1, :].shape[0]):
    gt_i = A_gt_deconv.iloc[index1, :].values[i]
    pred_i = A_dec_pred1[index1, :][i]
    if np.std(gt_i) < 1e-8 or np.std(pred_i) < 1e-8:
        r = 0.0
        print(1)
    else:
        r, _ = pearsonr(gt_i, pred_i)
        r = float(r) if np.isfinite(r) else 0.0
    pcc_list.append(r)

print(float(np.mean(pcc_list)))


# ============================================================
# Optimize A with the spatial adjacency graph
# ============================================================

def generate_optimized_A_softmax_normalized(A0, adj_matrix, lambda_reg=10.0, lr=0.05, epochs=1000):
    """
    Version using the normalized Laplacian matrix.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    A0_tensor = torch.tensor(A0, dtype=torch.float32).to(device)
    W = torch.tensor(adj_matrix, dtype=torch.float32).to(device)

    D_vec = torch.sum(W, dim=1)

    eps = 1e-8
    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D_vec + eps))

    I = torch.eye(W.shape[0]).to(device)
    L_sym = I - torch.mm(torch.mm(D_inv_sqrt, W), D_inv_sqrt)

    eps_log = 1e-8
    S_init = torch.log(A0_tensor)
    S = torch.nn.Parameter(S_init + torch.randn_like(S_init) * 0.01)

    optimizer = torch.optim.Adam([S], lr=lr)

    print(f"Starting optimization (using normalized Laplacian, Device: {device})...")

    for epoch in range(epochs):
        optimizer.zero_grad()

        A = torch.softmax(S, dim=1)

        norm_A = torch.norm(A, dim=1, keepdim=True)
        norm_A0 = torch.norm(A0_tensor, dim=1, keepdim=True)
        norm_A = torch.clamp(norm_A, min=1e-8)
        norm_A0 = torch.clamp(norm_A0, min=1e-8)

        dot_product = torch.sum(A * A0_tensor, dim=1, keepdim=True)
        cos_sim = dot_product / (norm_A * norm_A0)
        loss_sim = torch.mean(1 - cos_sim)

        loss_smooth = torch.trace(torch.mm(torch.mm(A.t(), L_sym), A))

        loss = loss_sim + lambda_reg * loss_smooth

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(
                f"Epoch {epoch+1}, Loss: {loss.item():.4f}, "
                f"Sim: {loss_sim.item():.4f}, Smooth: {loss_smooth.item():.4f}"
            )

    A_final = torch.softmax(S, dim=1)
    return A_final.detach().cpu().numpy()


Count_cell = adata_st.obsm['spatial']

coords = Count_cell

knn = NearestNeighbors(n_neighbors=3, metric='euclidean')
knn.fit(coords)

distances, indices = knn.kneighbors(coords)

adj_matrix = np.zeros((coords.shape[0], coords.shape[0]))

for i in range(coords.shape[0]):
    adj_matrix[i, indices[i]] = 1

adj_matrix = np.maximum(adj_matrix, adj_matrix.T)

print("Adjacency matrix shape:", adj_matrix.shape)
print("Number of non-zero elements:", np.count_nonzero(adj_matrix))

A0_data = A_dec_pred1.copy()
adj_data = adj_matrix.copy()

A_optimized = generate_optimized_A_softmax_normalized(
    A0_data, adj_data, lambda_reg=0.01, epochs=5000
)

print("Optimization complete, A shape:", A_optimized.shape)

pcc_list = []
for i in range(A_gt_deconv.iloc[:, :].shape[0]):
    gt_i = A_gt_deconv.iloc[:, :].values[i]
    pred_i = A_optimized[:, :][i]
    if np.std(gt_i) < 1e-8 or np.std(pred_i) < 1e-8:
        r = 0.0
        print(1)
    else:
        r, _ = pearsonr(gt_i, pred_i)
        r = float(r) if np.isfinite(r) else 0.0
    pcc_list.append(r)

print(float(np.mean(pcc_list)))

pd.DataFrame(A_optimized, columns=cell_types_gt_order).to_csv('ours/merfish_A_optimized_0425.csv')
pd.DataFrame(A_dec_pred1, columns=cell_types_gt_order).to_csv('ours/merfish_A_pred_0425.csv')
pd.DataFrame(pcc_list, index=A_gt_deconv.index).to_csv('ours/pcc_list_final')


# ============================================================
# Deconvolution evaluation function
# ============================================================

def evaluate_deconvolution(pred, true, eps=1e-12):
    """
    Evaluate spatial deconvolution predictions against ground truth using multiple metrics.
    """
    pred = np.asarray(pred)
    true = np.asarray(true)
    assert pred.shape == true.shape, "Prediction and ground truth matrices have inconsistent shapes"

    n_spots, n_types = pred.shape

    spot_pcc = []
    spot_spearman = []
    for i in range(n_spots):
        p = pred[i, :]
        t = true[i, :]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            spot_pcc.append(pcc)
            spot_spearman.append(spearman)

    mean_spot_pcc = np.nanmean(spot_pcc) if spot_pcc else np.nan
    mean_spot_spearman = np.nanmean(spot_spearman) if spot_spearman else np.nan

    type_pcc = []
    type_spearman = []
    for j in range(n_types):
        p = pred[:, j]
        t = true[:, j]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            type_pcc.append(pcc)
            type_spearman.append(spearman)

    mean_type_pcc = np.nanmean(type_pcc) if type_pcc else np.nan
    mean_type_spearman = np.nanmean(type_spearman) if type_spearman else np.nan

    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)

    jsd_list = []
    for i in range(n_spots):
        p = pred[i, :]
        t = true[i, :]
        p = np.maximum(p, 0)
        t = np.maximum(t, 0)
        p = p / (np.sum(p) + eps)
        t = t / (np.sum(t) + eps)
        jsd = jensenshannon(p, t, base=2) ** 2
        jsd_list.append(jsd)

    mean_jsd = np.mean(jsd_list)

    results = {
        'metric': [
            'mean_spot_pcc',
            'mean_spot_spearman',
            'mean_type_pcc',
            'mean_type_spearman',
            'overall_mse',
            'overall_rmse',
            'mean_jsd'
        ],
        'value': [
            mean_spot_pcc,
            mean_spot_spearman,
            mean_type_pcc,
            mean_type_spearman,
            mse,
            rmse,
            mean_jsd
        ]
    }

    df = pd.DataFrame(results)
    df.set_index('metric', inplace=True)
    return df


A_optimized1 = A_optimized.copy()
A_optimized1[A_optimized1 < 0.045] = 0
A_optimized1 = A_optimized1 / A_optimized1.sum(axis=1, keepdims=True)

df_results_ours = evaluate_deconvolution(A_optimized1, A_gt_deconv.values)
df_results_ours0 = evaluate_deconvolution(A_optimized, A_gt_deconv.values)

print(df_results_ours0)
print(df_results_ours)